<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_08_functions/notes_functions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Конспект: Функції та Функціональна Абстракція

> Урок 9 — повний конспект з теорією, чотирма функціональними патернами,
> прикладами та типовими помилками.

---

## 📋 Зміст

| # | Тема | Ключові концепції |
|---|------|-------------------|
| 1 | Навіщо функції? | DRY, Абстракція, Декомпозиція |
| 2 | Ментальна модель виконання | Stack Frame, локальні змінні |
| 3 | Параметри vs Аргументи | сигнатура, виклик |
| 4 | `return` vs `print` | передача даних у програму |
| 5 | Чисті функції | Pure Functions, Side Effects |
| 6 | `*args` і `**kwargs` | tuple / dict, розпаковка `*` `**`, 5 видів параметрів, пастки |
| 7 | Патерн: Предикат | фільтрація, `filter()` |
| 8 | Патерн: Трансформер | `map()`, comprehension |
| 9 | Патерн: Редьюсер | `sum()`, `max()`, `reduce()` |
| 10 | Функціональна Декомпозиція | divide and conquer |
| 11 | Повний Pipeline | Filter → Map → Reduce |
| 12 | Типові помилки | застереження, виправлення |
| 13 | Шпаргалка | швидкий довідник |
| 14 | Задачі | самостійна робота |

---

## 🧠 Як мислить програміст

Програмісти не пишуть довгі послідовності інструкцій — вони проектують **систему перетворення даних**.
Функції — це не просто «згрупований код», це **інструмент управління складністю**.

Коли програміст бачить нову задачу, він ставить чотири питання:

```
1. PREDICATE   — Що відфільтрувати? (Так / Ні)
2. TRANSFORMER — Як змінити форму даних? (один → інший)
3. REDUCER     — Як зібрати у кінцевий результат? (багато → одне)
4. DECOMPOSE   — Як розбити задачу на менші частини?
```

Увесь цей урок — про ці чотири патерни та інструменти Python для їх реалізації.

---

## 🏗️ Частина 1: Навіщо потрібні функції?

### Дві проблеми, які вирішують функції

**Проблема 1 — Дублювання коду (WET: Write Everything Twice)**

```python
# БЕЗ функцій — та сама логіка скрізь
bill1 = 150.0
tax1  = bill1 * 0.20   # ← якщо ставку змінити на 0.22,
final1 = bill1 + tax1  #   треба виправляти у 10 місцях!

bill2 = 300.0
tax2  = bill2 * 0.20
final2 = bill2 + tax2
```

**Проблема 2 — Когнітивне навантаження**

Читаючи 300 рядків коду, мозок не може утримувати всі деталі одночасно.
Функції дозволяють **думати на рівні понять**, а не деталей синтаксису.

### Три рішення, що дають функції

| Проблема | Рішення | Принцип |
|---|---|---|
| Дублювання | Одна функція → багато викликів | **DRY** (Don't Repeat Yourself) |
| Складність | Функція = «чорний ящик» | **Абстракція** |
| Великий розмір | Велика задача → маленькі функції | **Декомпозиція** |

> **Абстракція** — усунення неважливого і посилення суттєвого.
> Знаючи `apply_tax(bill)`, вам не потрібно думати про множення.
> Ви думаєте про **що** відбувається, а не **як**.

In [1]:
# Без функцій — WET (Write Everything Twice)
bill1 = 150.0
tax1  = bill1 * 0.20
final1 = bill1 + tax1

bill2 = 300.0
tax2  = bill2 * 0.20   # та сама логіка — дублювання!
final2 = bill2 + tax2

print(f"WET: final1={final1}, final2={final2}")
print()

# З функцією — DRY (Don't Repeat Yourself)
def apply_tax(bill, rate=0.20):
    """Повертає суму з податком. Логіка в одному місці."""
    return bill * (1 + rate)

final1 = apply_tax(150.0)
final2 = apply_tax(300.0)
final3 = apply_tax(500.0, rate=0.15)  # іншу ставку — просто змінюємо аргумент

print(f"DRY: final1={final1}, final2={final2}, final3={final3}")
print()

# Тепер хочемо змінити ставку — тільки одне місце!
print("Якщо треба змінити ставку → змінюємо тільки в одному місці: def apply_tax(bill, rate=0.22)")

WET: final1=180.0, final2=360.0

DRY: final1=180.0, final2=360.0, final3=575.0

Якщо треба змінити ставку → змінюємо тільки в одному місці: def apply_tax(bill, rate=0.22)


---

## ⚙️ Частина 2: Ментальна модель виконання — Stack Frame

### Синтаксис функції

```
def  назва_функції  ( параметр1, параметр2 ) :  ← сигнатура
     │                │
  ключове слово    параметри (placeholder-и, НЕ реальні дані)

    # тіло — код, що виконується
    result = параметр1 + параметр2
    return result   ← повертаємо значення
```

### Що відбувається при виклику (Stack Frame)

```
1. ВИКЛИК:   add(5, 10)         ← аргументи передаються в функцію
                ↓
2. СТЕК:     Python створює тимчасовий «ящик» у пам'яті
             ┌─────────────────────┐
             │  a = 5              │ ← локальні змінні живуть тут
             │  b = 10             │
             │  result = 15        │
             └─────────────────────┘
                ↓
3. RETURN:   повертає 15 у головну програму
             «ящик» знищується → a, b, result зникають назавжди
```

> **Важливо:** локальні змінні існують **тільки під час** виконання функції.
> Ззовні вони невидимі — це й називається **ізоляція (scope)**.

In [28]:
def myfunc(a):
    """my function"""

    result = 1 + a
    return float(result)

myfunc(10)



In [4]:
# Демонстрація Stack Frame та ізоляції
def add(a, b):
    result = a + b          # result — локальна змінна
    print(f"  [всередині] a={a}, b={b}, result={result}")
    return result
    # після return: а, b, result зникають


print("Виклик add(5, 10):")
answer = add(5, 10)
print(f"  [ззовні] answer={answer}")
print()

# Локальні змінні ззовні не видно
try:
    print(result)   # ← result не існує ззовні!
except NameError as e:
    print(f"NameError: {e}")
    print("  Локальні змінні ізольовані — ззовні невидимі!")
print()

# Кожен виклик — свій незалежний Stack Frame
print("Два незалежних виклики:")
r1 = add(2, 3)    # свій frame: a=2, b=3
r2 = add(10, 20)  # свій frame: a=10, b=20
print(f"  r1={r1}, r2={r2}")

Виклик add(5, 10):
  [всередині] a=5, b=10, result=15
  [ззовні] answer=15

1

Два незалежних виклики:
  [всередині] a=2, b=3, result=5
  [всередині] a=10, b=20, result=30
  r1=5, r2=30


---

## 🔤 Частина 3: Параметри vs Аргументи

Ці два терміни часто плутають. Вони описують **різні моменти** в житті функції:

```
ПАРАМЕТРИ — імена у визначенні функції (при написанні коду)
              ↓           ↓
def describe(name,      age):
    ...                          ← placeholder-и, «конверти без листів»

АРГУМЕНТИ — реальні дані при виклику (при запуску)
              ↓          ↓
describe("Alice",      25)       ← реальні значення, «листи в конверти»
```

### Типи параметрів

| Тип | Синтаксис | Особливість |
|-----|-----------|-------------|
| Позиційний | `def f(a, b)` | Порядок важливий |
| Зі значенням | `def f(a, b=10)` | Необов'язковий |
| Keyword-only | `def f(a, *, kw)` | Тільки через `kw=значення` |

In [36]:
# Позиційні параметри — порядок важливий!
def describe(name, age):
    print(f"  {name} має {age} років")


describe(name="Alice", age=25)   # a="Alice", b=25 — правильно
describe(25, "Alice")   # a=25, b="Alice" — семантично неправильно!
print()

# Параметри зі значенням за замовчуванням
def greet(name, greeting="Привіт"):
    print(f"  {greeting}, {name}!")

greet("Олена")              # greeting = "Привіт" за замовчуванням
greet("Іван", "Доброго дня")  # переопределяємо
print()

# Keyword-аргументи — ім'я явно
def power(base, exponent=2):
    return base ** exponent

print(f"  power(3)          = {power(3)}")
print(f"  power(3, 3)       = {power(3, 3)}")
print(f"  power(exponent=3, base=2) = {power(exponent=3, base=2)}")  # keyword — порядок не важливий!

  Alice має 25 років
  25 має Alice років

  Привіт, Олена!
  Доброго дня, Іван!

  power(3)          = 9
  power(3, 3)       = 27
  power(exponent=3, base=2) = 8


---

## 🖨️ Частина 4: `return` vs `print` — критична різниця

Це одна з найпоширеніших помилок початківців.

```
print()  →  для ЛЮДИНИ   (показує на екрані, програма нічого не отримує)
return   →  для ПРОГРАМИ  (передає значення назад у пайплайн)
```

```python
# ПОМИЛКА: print замість return
def bad_add(a, b):
    print(a + b)     ← показує 5, але програма отримує None

result = bad_add(2, 3)
print(result)        ← виведе: None
result * 10          ← TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
```

> Якщо функція не має `return` — Python **автоматично** повертає `None`.
> `None` — це «порожнє значення» (не `0`, не `""`, а **відсутність** значення).

In [37]:
# ПОМИЛКА: print замість return
def bad_add(a, b):
    print(a + b)   # показує 5 на екрані...


# ПРАВИЛЬНО: return
def good_add(a, b):
    return a + b   # передає значення в програму


print("--- bad_add(2, 3) ---")
bad_result = bad_add(2, 3)       # виводить 5 під час виконання
print(f"bad_result  = {bad_result!r}")  # None!

print()
print("--- good_add(2, 3) ---")
good_result = good_add(2, 3)     # нічого не виводить, але зберігає 5
print(f"good_result = {good_result!r}")  # 5
print(f"good_result * 10 = {good_result * 10}")  # 50 — можна використати!

print()
# bad_result * 10 → TypeError
try:
    _ = bad_result * 10
except TypeError as e:
    print(f"bad_result * 10 → TypeError: {e}")
    print("  None не можна множити — функція не повернула значення!")

--- bad_add(2, 3) ---
5
bad_result  = None

--- good_add(2, 3) ---
good_result = 5
good_result * 10 = 50

bad_result * 10 → TypeError: unsupported operand type(s) for *: 'NoneType' and 'int'
  None не можна множити — функція не повернула значення!


---

## 🏭 Частина 5: Чисті функції (Pure Functions)

### Два правила чистої функції

```
1. ДЕТЕРМІНОВАНІСТЬ:  однакові аргументи → ЗАВЖДИ однаковий результат
2. БЕЗ ПОБІЧНИХ ЕФЕКТІВ:  не змінює нічого ззовні (no side effects)
```

**Ментальна модель:** ідеальна машина на заводі.
Закладаєте однакову сировину → отримуєте однаковий продукт.
Машина не «забруднює цех» — не змінює глобальні змінні.

### Нечиста vs Чиста — порівняння

```python
# НЕЧИСТА — мутує вхідний список!
def add_value_impure(items, value):
    items.append(value)   ← змінює ОРИГІНАЛ
    return items

# ЧИСТА — створює НОВИЙ список
def add_value_pure(items, value):
    return items + [value]  ← оригінал залишається
```

### Аналогія з вбудованими функціями

| Функція | Тип | Що робить |
|---------|-----|-----------|
| `list.sort()` | Нечиста | Сортує список **на місці** (in-place) |
| `sorted()` | Чиста | Повертає **новий** відсортований список |
| `list.append()` | Нечиста | Додає елемент **в оригінал** |
| `list + [item]` | Чиста | Повертає **новий** список |

In [5]:
# === Демонстрація: нечиста vs чиста функція ===

# НЕЧИСТА: мутує вхідний список (side effect!)
def apply_discount_impure(prices, discount):

    for i in range(len(prices)):       # змінюємо ОРИГІНАЛЬНИЙ список
        prices[i] = prices[i] * (1 - discount)
    return prices


# ЧИСТА: повертає НОВИЙ список, оригінал не торкається
def apply_discount_pure(prices, discount):
    return [p * (1 - discount) for p in prices]   # новий список!


my_prices = [100.0, 200.0, 300.0]
print(f"До виклику:            {my_prices}")

result = apply_discount_impure(my_prices, 0.10)
print(f"Результат (нечиста):   {result}")
print(f"ОРИГІНАЛ ЗНИЩЕНИЙ:     {my_prices}")    # ← my_prices змінений!
print()

# Чиста функція — безпечна
my_prices2 = [100.0, 200.0, 300.0]
result2 = apply_discount_pure(my_prices2, 0.10)
print(f"Результат (чиста):     {result2}")
print(f"Оригінал збережений:   {my_prices2}")   # ← незмінений!

До виклику:            [100.0, 200.0, 300.0]
Результат (нечиста):   [90.0, 180.0, 270.0]
ОРИГІНАЛ ЗНИЩЕНИЙ:     [90.0, 180.0, 270.0]

Результат (чиста):     [90.0, 180.0, 270.0]
Оригінал збережений:   [100.0, 200.0, 300.0]


In [38]:
# Аналогія: sort() vs sorted()
data = [3, 1, 4, 1, 5, 9, 2, 6]

# ЧИСТА — оригінал збережений
sorted_data = sorted(data)
print(f"sorted() результат: {sorted_data}")
print(f"Оригінал після sorted(): {data}")   # незмінений!
print()

# НЕЧИСТА — оригінал змінений
data.sort()
print(f"Після .sort(): {data}")              # ← оригінал знищено

sorted() результат: [1, 1, 2, 3, 4, 5, 6, 9]
Оригінал після sorted(): [3, 1, 4, 1, 5, 9, 2, 6]

Після .sort(): [3, 1, 4, 1, 5, 9, 2, 6]


---

## 📦 Частина 6: `*args` і `**kwargs` — змінна кількість аргументів

```
def func(*args):
             ↑
   зірочка «пакує» всі позиційні аргументи у TUPLE
```

**Коли використовувати:**
- Кількість аргументів **невідома заздалегідь**
- Функція повинна прийняти **будь-яку кількість** значень

```python
process_sales(150)             → args = (150,)
process_sales(150, 300, 75)    → args = (150, 300, 75)
process_sales(1, 2, 3, 4, 5)  → args = (1, 2, 3, 4, 5)
```

**План частини:**
6.1 `*args` під мікроскопом · 6.2 `**kwargs` · 6.3 розпаковка `*` з list / tuple / set / str / range · 6.4 розпаковка `**dict` ·
6.5 розпаковка в присвоєнні та літералах · 6.6 прокидання `func(*args, **kwargs)` · 6.7 п'ять видів параметрів (`/`, `*`) ·
6.8 пастки · 6.9 чому Python влаштований саме так

In [39]:
# *args пакує всі аргументи у tuple
def process_sales(*args):
    print(f"  тип args: {type(args)}")
    print(f"  вміст:    {args}")
    print(f"  кількість: {len(args)}")
    return sum(args)

print("--- Один аргумент ---")
process_sales(150.0)

print("\n--- Три аргументи ---")
total = process_sales(150.0, 300.0, 75.5)
print(f"  Разом: {total}")

print("\n--- П'ять аргументів ---")
total = process_sales(150, -20, 300, 50, -10)
print(f"  Разом: {total}")

--- Один аргумент ---
  тип args: <class 'tuple'>
  вміст:    (150.0,)
  кількість: 1

--- Три аргументи ---
  тип args: <class 'tuple'>
  вміст:    (150.0, 300.0, 75.5)
  кількість: 3
  Разом: 525.5

--- П'ять аргументів ---
  тип args: <class 'tuple'>
  вміст:    (150, -20, 300, 50, -10)
  кількість: 5
  Разом: 470


In [50]:
# Практичний приклад: добуток довільної кількості чисел
def calculate_product(*args):
    """Множить усі передані числа між собою."""
    total = 1
    for number in args:      # args — це tuple, ітеруємо по ньому
        total *= number
    return total


print(calculate_product(2, 5))           # 10
print(calculate_product(2, 5, 3))        # 30
print(calculate_product(2, 5, 3, 2))     # 60
print(calculate_product(1, 2, 3, 4, 5))  # 120  ← 5!
print()

# Можна комбінувати *args з іншими параметрами
def weighted_sum(weight, *values):
    """Сума всіх values, помножена на weight."""
    return weight * sum(values)

print(f"weighted_sum(2, 10, 20, 30) = {weighted_sum(2, 10, 20, 30)}")  # 2*(10+20+30) = 120
w_sum = ( 10, 20, 30)
print(f"weighted_sum(2, 10, 20, 30) = {weighted_sum(2, *w_sum)}")  # 2*(10+20+30) = 120


10
30
60
120

weighted_sum(2, 10, 20, 30) = 120
weighted_sum(2, 10, 20, 30) = 120


### 6.1 `*args` під мікроскопом — що саме отримує функція

Зірочка в **оголошенні** `def f(*args)` означає «збери всі зайві позиційні аргументи в один **кортеж**».
Подивимось, що відбувається всередині: тип, довжина, індекси, цикл.

```
виклик:      log_args(10, "кава", True)
                       │    │      │
всередині:   args = (10, 'кава', True)   ← tuple, незмінний
```

In [ ]:
def log_args(*args):
    print(f"  [вхід]   args = {args!r}")
    print(f"  [тип]    {type(args).__name__}, len = {len(args)}")
    if args:
        print(f"  [індекс] args[0] = {args[0]!r}, args[-1] = {args[-1]!r}")
    for i, value in enumerate(args):
        print(f"  [цикл]   #{i}: {value!r} ({type(value).__name__})")


print("--- log_args(10, 'кава', True) ---")
log_args(10, "кава", True)

print("\n--- log_args() — без аргументів ---")
log_args()            # args = () — порожній кортеж, НЕ помилка

print("\n--- args — це tuple, його не можна змінити ---")
def try_modify(*args):
    try:
        args[0] = 999
    except TypeError as e:
        print(f"  TypeError: {e}")

try_modify(1, 2, 3)

### 6.2 `**kwargs` — довільні **іменовані** аргументи → словник

```
def func(**kwargs):
              ↑↑
   дві зірочки «пакують» усі зайві  ключ=значення  у DICT
```

| | `*args` | `**kwargs` |
|---|---|---|
| що збирає | зайві **позиційні** аргументи | зайві **іменовані** аргументи `key=value` |
| тип усередині | `tuple` | `dict` |
| порожнє значення | `()` | `{}` |

- Порядок ключів у `kwargs` = порядок, у якому їх передали при виклику (гарантовано з Python 3.7).
- Імена `args` і `kwargs` — лише **домовленість**. Працюють саме `*` і `**`: `def f(*items, **options)` — те саме.

In [ ]:
def log_kwargs(**kwargs):
    print(f"  [вхід]  kwargs = {kwargs!r}")
    print(f"  [тип]   {type(kwargs).__name__}, len = {len(kwargs)}")
    print(f"  [ключі] {list(kwargs.keys())}")
    for key, value in kwargs.items():
        print(f"  [пара]  {key} -> {value!r}")
    # .get() — безпечне читання необов'язкового ключа
    print(f"  [get]   sugar = {kwargs.get('sugar', 0)}  (0 — якщо не передали)")


print("--- log_kwargs(size='L', milk=True, sugar=2) ---")
log_kwargs(size="L", milk=True, sugar=2)

print("\n--- log_kwargs(milk=False, size='S') — інший порядок ---")
log_kwargs(milk=False, size="S")


print("\n--- *args і **kwargs разом ---")
def show_details(*args, **kwargs):
    print("  Позиційні (args):", args)      # tuple
    print("  Іменовані (kwargs):", kwargs)  # dict

show_details(1, 2, 3, name="Олексій", role="admin")


print("\n--- Імена — лише домовленість ---")
def make_order(*items, **options):
    print(f"  items   = {items}")
    print(f"  options = {options}")

make_order("кава", "круасан", takeaway=True)

### 6.3 Розпаковка `*` **при виклику** — з будь-якої колекції

Та сама зірочка, але **в іншому місці** робить протилежне:

```
def f(*args)      ← в ОГОЛОШЕННІ:  багато аргументів  →  один tuple   (пакування)
f(*collection)    ← у ВИКЛИКУ:     одна колекція      →  багато аргументів (розпаковка)
```

`*` працює з **будь-яким ітерованим об'єктом**: `list`, `tuple`, `set`, `str`, `range`, генератор, `dict` (віддає ключі).

⚠️ **Множина (`set`) не має порядку.** Для невеликих цілих чисел CPython зазвичай віддає їх «по зростанню», але для рядків порядок може змінюватися **між запусками**. Не розпаковуйте `set` у функцію, де порядок важливий.

In [ ]:
def receive(a, b, c):
    print(f"    отримала: a={a!r}, b={b!r}, c={c!r}")


def show(label, collection):
    print(f"--- {label}: {collection!r} ---")
    receive(*collection)


show("list ", [1, 2, 3])
show("tuple", (10, 20, 30))
show("set  ", {3, 1, 2})          # порядок множини НЕ гарантований!
show("str  ", "abc")              # рядок — це послідовність символів
show("range", range(100, 103))
show("dict ", {"x": 1, "y": 2, "z": 3})   # * з dict віддає лише КЛЮЧІ

print("--- генератор ---")
receive(*(n * n for n in range(1, 4)))

print("\n--- Розпаковку можна поєднувати зі звичайними аргументами ---")
middle = [2, 3]
print("  print(1, *middle, 4) →", end=" ")
print(1, *middle, 4)

print("\n--- Кількість елементів має збігатися з параметрами ---")
try:
    receive(*[1, 2])
except TypeError as e:
    print(f"  TypeError: {e}")
try:
    receive(*[1, 2, 3, 4])
except TypeError as e:
    print(f"  TypeError: {e}")

print("\n--- А в функцію з *args можна розпакувати скільки завгодно ---")
def total(*prices):
    print(f"    prices = {prices}")
    return sum(prices)

bills = [150, 300, 75]
print(f"  total(*bills) = {total(*bills)}")
print(f"  total(bills)  — БЕЗ зірочки передається ОДИН аргумент-список:")
try:
    total(bills)
except TypeError as e:
    print(f"  TypeError: {e}")

### 6.4 Розпаковка `**dict` при виклику

`**` перетворює пари `ключ: значення` на іменовані аргументи `ключ=значення`.

```python
config = {"c": 3, "d": 4}
process_data(1, 2, **config)   # те саме, що process_data(1, 2, c=3, d=4)
```

Правила:
- ключі словника мають бути **рядками** й збігатися з іменами параметрів;
- зайвий ключ → `TypeError: ... got an unexpected keyword argument`;
- відсутній обов'язковий → `TypeError: ... missing ... required ...`;
- ключ, що дублює вже переданий позиційно аргумент → `TypeError: ... got multiple values for argument`.

In [ ]:
def process_data(a, b, c, d):
    print(f"    a={a}, b={b}, c={c}, d={d}")


coords = (1, 2)
config = {"c": 3, "d": 4}

print("--- process_data(*coords, **config) ---")
print(f"  coords = {coords}  →  a, b")
print(f"  config = {config}  →  c=3, d=4")
process_data(*coords, **config)

print("\n--- Весь виклик зі словника ---")
full = {"d": 40, "a": 10, "c": 30, "b": 20}   # порядок ключів не важливий
process_data(**full)

print("\n--- Помилки розпаковки ---")
cases = {
    "зайвий ключ 'e'":        {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5},
    "бракує ключа 'd'":       {"a": 1, "b": 2, "c": 3},
}
for label, kw in cases.items():
    try:
        process_data(**kw)
    except TypeError as e:
        print(f"  {label}: TypeError: {e}")

try:
    process_data(1, 2, 3, **{"a": 100, "d": 4})   # a вже прийшов позиційно
except TypeError as e:
    print(f"  конфлікт 'a': TypeError: {e}")


print("\n--- **dict у функцію з **kwargs ---")
def make_coffee(size, **extras):
    print(f"    size={size!r}, extras={extras}")

defaults = {"milk": True, "sugar": 1}
make_coffee("M", **defaults)
make_coffee("L", **defaults, sugar_free_syrup="ваніль")

### 6.5 Розпаковка поза функціями: присвоєння та літерали колекцій

Зірочки працюють не лише в `def` і виклику:

| Запис | Що робить |
|---|---|
| `first, *rest = [1, 2, 3, 4]` | `first = 1`, `rest = [2, 3, 4]` (завжди **list**) |
| `first, *_, last = data` | взяти крайні, середину — «викинути» в `_` |
| `[*a, *b]` | склеїти будь-які ітеровані в **новий список** |
| `(*a, *b)` | … у новий кортеж |
| `{*a, *b}` | … у нову множину (дублікати зникнуть) |
| `{**d1, **d2}` | злити словники; при однакових ключах **перемагає правий** |

In [ ]:
data = [10, 20, 30, 40, 50]

first, *rest = data
print(f"first, *rest = {data}")
print(f"  first = {first}, rest = {rest}  ({type(rest).__name__})")

*init, last = data
print(f"*init, last  → init = {init}, last = {last}")

first, *_, last = data
print(f"first, *_, last → first = {first}, last = {last}, _ = {_}")

a, *empty, b = [1, 2]
print(f"a, *empty, b = [1, 2] → empty = {empty}  (порожній список, не помилка)")

print("\n--- Склеювання різних колекцій ---")
lst = [1, 2]
tpl = (3, 4)
st  = {5, 6}
s   = "ab"
print(f"[*lst, *tpl, *st, *s] = {[*lst, *tpl, *st, *s]}")
print(f"(*lst, *tpl)          = {(*lst, *tpl)}")
print(f"{{*lst, *tpl, *lst}}    = {({*lst, *tpl, *lst})}   ← дублікати зникли")

print("\n--- Злиття словників ---")
base   = {"size": "M", "milk": False, "sugar": 1}
custom = {"milk": True, "syrup": "карамель"}
merged = {**base, **custom}
print(f"base   = {base}")
print(f"custom = {custom}")
print(f"merged = {merged}   ← milk взято з custom (правий перемагає)")
print(f"base після злиття: {base}  ← оригінал не змінився")

print("\n--- print(*items) — кожен елемент окремим аргументом ---")
items = ["кава", "чай", "сік"]
print(items)
print(*items)
print(*items, sep=" | ")

### 6.6 Прокидання аргументів далі: `func(*args, **kwargs)`

Пакування в `def` + розпаковка у виклику = функція-**посередник**, яка приймає **будь-яку** сигнатуру й передає її далі без змін.
Це фундамент **декораторів** (урок 10) і `super().__init__(*args, **kwargs)` у класах.

```
logger_wrapper(func, 2, 3, sep="-")
                     │  │   │
           args = (2, 3)   kwargs = {'sep': '-'}      ← пакування
                     ↓
           func(*args, **kwargs)  →  func(2, 3, sep="-")  ← розпаковка
```

In [ ]:
def logger_wrapper(func, *args, **kwargs):
    print(f"  [LOG] виклик {func.__name__} з args={args} kwargs={kwargs}")
    result = func(*args, **kwargs)
    print(f"  [LOG] {func.__name__} повернула {result!r}")
    return result


def price_with_service(bill, service=10):
    return bill + bill * service / 100


logger_wrapper(price_with_service, 500.0)
logger_wrapper(price_with_service, 500.0, service=5)
logger_wrapper(max, 3, 9, 4)
logger_wrapper(sorted, [3, 1, 2], reverse=True)

### 6.7 П'ять видів параметрів у `def`

Спершу кожен вид окремо, на маленькому прикладі. **Повну схему подивимось наприкінці**, коли кожен її елемент уже буде зрозумілий.

| # | Вид | Як передавати | Тип усередині |
|---|---|---|---|
| 1 | тільки за порядком (*positional-only*) — перед `/` | лише за порядком | значення |
| 2 | за порядком або за іменем (*positional-or-keyword*) | як завгодно | значення |
| 3 | зайві без імені (*var-positional*) — `*args` | збираються | `tuple` |
| 4 | тільки за іменем (*keyword-only*) — після `*` або `*args` | лише `key=value` | значення |
| 5 | зайві з іменем (*var-keyword*) — `**kwargs`, завжди останній | збираються | `dict` |

**Значення за замовчуванням — не окремий вид.** Його може мати і вид 2 (`sugar=1`), і вид 4 (`timeout=30` після `*`).
Жоден вид не обов'язковий, але **порядок тих, що є, змінювати не можна** — інакше `SyntaxError` ще до запуску.

#### 1. Тільки за порядком: `/`

Звичайну функцію `calculate_speed(distance, time)` можна викликати і `calculate_speed(100, 2)`, і `calculate_speed(distance=100, time=2)`.
Додамо в оголошення `/` — і другий спосіб зникне.

- **Символ `/` у списку параметрів означає: усі параметри перед ним передаються тільки за порядком** (Python 3.8+, PEP 570).
- У рядку `def` символ `/` **нічого не ділить** і **не є параметром**: третього значення передавати не треба. У виклику `/` не пишуть.
- `/` посередині обмежує лише параметри **ліворуч**: у `def f(distance, /, time)` параметр `time` можна передати і за порядком, і за іменем. «Після `/`» ≠ «тільки за іменем».
- **Навіщо:** автор може перейменувати такі параметри, не зламавши чужий код — за іменем до них ніхто не звертався. Для початківця головне — **прочитати** запис у документації: `len(obj, /)` означає `len([1, 2])`, але не `len(obj=[1, 2])`.

In [ ]:
def calculate_speed(distance, time, /):   # / — роздільник, а не ділення
    print(f"    distance={distance}, time={time}")
    return distance / time                 # / — ділення


print("calculate_speed(100, 2) →", calculate_speed(100, 2))
try:
    calculate_speed(distance=100, time=2)
except TypeError as e:
    print(f"calculate_speed(distance=100, time=2) → TypeError: {e}")

print("\n--- / посередині обмежує лише ліву частину ---")
def multiply(a, /, b):
    return a * b

for label, call in {
    "multiply(3, 4)":     lambda: multiply(3, 4),
    "multiply(3, b=4)":   lambda: multiply(3, b=4),
    "multiply(a=3, b=4)": lambda: multiply(a=3, b=4),
}.items():
    try:
        print(f"{label:<20} → {call()}")
    except TypeError as e:
        print(f"{label:<20} → TypeError: {e}")

print("\n--- Вбудовані функції: len(obj, /) ---")
print("len([1, 2, 3]) →", len([1, 2, 3]))
try:
    len(obj=[1, 2, 3])
except TypeError as e:
    print(f"len(obj=[1, 2, 3]) → TypeError: {e}")

#### `/` заборонив імена — як їх повернути?

```python
def calculate_speed(distance, time, /):
    return distance / time
```

Цей запис означає: «передавай два значення **за порядком**, без `distance=` і `time=`».

- Імена `distance` і `time` лишаються **всередині функції** — тому працює `return distance / time`. Заборонено лише використовувати їх **у виклику** для передачі аргументів.
- Хочеш дозволити обидва способи — **прибери `/` і повторно виконай клітинку з `def`**. Поки нове оголошення не виконано, Python пам'ятає стару версію з `/`.
- `/` ставлять **лише в оголошенні `def`**. У виклику його не пишуть.

Нижче — три оголошення поруч (з різними іменами, щоб порівняти):

| Оголошення | Що дозволено |
|---|---|
| `def f(distance, time):` | обидва параметри — за порядком або за іменем |
| `def f(distance, /, time):` | `distance` — тільки за порядком; `time` — обома способами |
| `def f(distance, time, /):` | обидва — тільки за порядком |

Для звичайної функції обчислення швидкості `/` не потрібен — спокійно обходьтеся без нього.

In [ ]:
# Три оголошення — три різні правила виклику
def speed_any(distance, time):          # без /
    return distance / time

def speed_mid(distance, /, time):       # / посередині
    return distance / time

def speed_pos(distance, time, /):       # / наприкінці
    return distance / time


for func in [speed_any, speed_mid, speed_pos]:
    print(f"--- {func.__name__} ---")
    print(f"    (100, 2)                → {func(100, 2)}")
    try:
        print(f"    (100, time=2)           → {func(100, time=2)}")
    except TypeError as e:
        print(f"    (100, time=2)           → TypeError: {e}")
    try:
        print(f"    (distance=100, time=2)  → {func(distance=100, time=2)}")
    except TypeError as e:
        print(f"    (distance=100, time=2)  → TypeError: {e}")

#### 2. За порядком або за іменем (Positional or Keyword)

- Звичайні параметри — після `/` (якщо він є) і до `*args` або `*`. Якщо в `def` немає ні `/`, ні `*`, усі параметри саме такі.
- Можна передати **за порядком** або **за іменем** (тоді порядок не важливий), або змішано — спершу без імені.
- Вони можуть бути **обов'язковими** (`name`) або мати **значення за замовчуванням** (`msg="Привіт"`) — див. наступну клітинку.

In [ ]:
def greet(name, msg):
    print(f"    name={name!r}, msg={msg!r}  →  {msg}, {name}!")


print("greet('Олексій', 'Привіт')          # позиційно")
greet("Олексій", "Привіт")

print("greet(msg='Вітаю', name='Олена')    # за іменем, порядок не важливий")
greet(msg="Вітаю", name="Олена")

print("greet('Тарас', msg='Добрий день')   # змішано: спершу позиційні")
greet("Тарас", msg="Добрий день")

#### Окремо: значення за замовчуванням (не окремий вид!)

- `param=value` — аргумент можна пропустити, тоді береться значення за замовчуванням.
- Серед параметрів, які можна передати за порядком, ті, що мають значення за замовчуванням, стоять **після** обов'язкових: `def f(sugar=1, size)` → `SyntaxError`.
- Параметри «тільки за іменем» (після `*`) теж можуть мати значення за замовчуванням, і там такого обмеження немає: `def f(*, sugar=1, size)` — коректно.
- Значення обчислюється **один раз** при виконанні `def` → пастка зі списком (6.8, Пастка 1).

In [ ]:
def make_coffee(size, sugar=1):
    print(f"    Кава {size}, цукор: {sugar}")


print("make_coffee('L')            # sugar = 1 за замовчуванням")
make_coffee("L")
print("make_coffee('M', sugar=2)   # sugar переозначено")
make_coffee("M", sugar=2)
print("make_coffee('S', 0)         # можна й позиційно")
make_coffee("S", 0)

# Де Python зберігає значення за замовчуванням:
print(f"\nmake_coffee.__defaults__ = {make_coffee.__defaults__}")

# Неправильний порядок — помилка ще до запуску (compile перевіряє синтаксис)
try:
    compile("def f(sugar=1, size): pass", "<demo>", "exec")
except SyntaxError as e:
    print(f"def f(sugar=1, size) → SyntaxError: {e.msg}")

compile("def f(*, sugar=1, size): pass", "<demo>", "exec")
print("def f(*, sugar=1, size) → OK: тільки іменовані можуть іти в будь-якому порядку")

#### 3. Зайві без імені: `*args` (і поодинока `*`)

- **`*args`** — збирає всі **зайві** позиційні аргументи в `tuple`. Звичайні параметри перед ним забирають свої значення **першими**.
- **Поодинока `*`** — нічого не збирає, лише ставить «стіну»: позиційні аргументи закінчились, далі — тільки іменовані (категорія 5).

In [ ]:
def multiply_all(factor, *numbers):
    # factor бере ПЕРШИЙ аргумент, numbers збирає ВСІ інші у tuple
    print(f"    factor={factor}, numbers={numbers}")
    return [x * factor for x in numbers]


print("multiply_all(2, 1, 2, 3) →", multiply_all(2, 1, 2, 3))
print("multiply_all(10)         →", multiply_all(10))   # numbers = ()


print("\n--- Поодинока * : стіна між позиційними та іменованими ---")
def connect(host, port, *, timeout=30):
    print(f"    host={host}, port={port}, timeout={timeout}")

connect("localhost", 8080)
connect("localhost", 8080, timeout=10)
try:
    connect("localhost", 8080, 10)
except TypeError as e:
    print(f"connect('localhost', 8080, 10) → TypeError: {e}")

#### 4. Тільки за іменем (Keyword-Only)

- **Синтаксис:** усі параметри **після `*args` або поодинокої `*`**.
- **Правило:** передати позиційно **неможливо** — тільки `key=value`.
- Можуть бути **обов'язковими** (без значення за замовчуванням) або необов'язковими.
- **Навіщо:** прапорці та налаштування читаються однозначно. `send_email(to, subject="Новини", secure=True)` зрозуміліше, ніж `send_email(to, "Новини", True)`.

In [ ]:
def send_email(to, *, subject="Без теми", secure=True):
    print(f"    to={to!r}, subject={subject!r}, secure={secure}")


send_email("test@mail.com", subject="Новини", secure=True)
send_email("test@mail.com")

try:
    send_email("test@mail.com", "Новини", True)
except TypeError as e:
    print(f"send_email(..., 'Новини', True) → TypeError: {e}")


print("\n--- Обов'язковий keyword-only (без значення за замовчуванням) ---")
def report(*orders, currency):
    print(f"    orders={orders}, currency={currency!r}")

report(150, 300, currency="UAH")
try:
    report(150, 300)
except TypeError as e:
    print(f"report(150, 300) → TypeError: {e}")

print(f"\nsend_email.__kwdefaults__ = {send_email.__kwdefaults__}")

#### 5. Зайві з іменем: `**kwargs`

- **Синтаксис:** `**kwargs` — завжди **в самому кінці** сигнатури.
- **Правило:** збирає всі іменовані аргументи, які **не перехопив** жоден інший параметр, у `dict`. Порядок ключів = порядок передачі.

In [ ]:
def build_profile(first, last, **user_info):
    print(f"    first={first!r}, last={last!r}")
    print(f"    user_info (до)    = {user_info}")
    user_info["first_name"] = first
    user_info["last_name"] = last
    print(f"    user_info (після) = {user_info}")
    return user_info


profile = build_profile("Іван", "Петренко", age=30, city="Київ")
print(f"profile = {profile}")

print("\n--- first/last НЕ потрапляють у kwargs, навіть якщо передані за іменем ---")
build_profile(last="Коваль", first="Олена", city="Львів")

print("\n--- **kwargs не на останньому місці — SyntaxError ---")
for bad in ["def f(**kw, a): pass", "def f(**kw, *args): pass"]:
    try:
        compile(bad, "<demo>", "exec")
    except SyntaxError as e:
        print(f"  {bad:<28} → SyntaxError: {e.msg}")

#### Повна картина — тепер, коли кожен елемент знайомий

```python
def example(pos_only, /, pos_or_kw, default_pos=10, *args, kw_only, kw_default=20, **kwargs):
```

```
 pos_only , /  pos_or_kw , default_pos=10 , *args , kw_only , kw_default=20 , **kwargs
 └──1───┘     └──────────2──────────────┘   └─3─┘   └──────────4─────────┘   └───5───┘
 за порядком  за порядком або за іменем     зайві   тільки за іменем          зайві з іменем
```

Запустіть клітинку й простежте, куди потрапляє кожен аргумент.
Потім змініть виклик (приберіть аргумент, передайте `pos_only` за іменем, `kw_only` — за порядком) і **передбачте** результат до запуску.

In [ ]:
def example(pos_only, /, pos_or_kw, default_pos=10, *args, kw_only, kw_default=20, **kwargs):
    print(f"  [1] pos_only    = {pos_only!r}")
    print(f"  [2] pos_or_kw   = {pos_or_kw!r}")
    print(f"  [2] default_pos = {default_pos!r}")
    print(f"  [3] args        = {args!r}")
    print(f"  [4] kw_only     = {kw_only!r}, kw_default = {kw_default!r}")
    print(f"  [5] kwargs      = {kwargs!r}")


print("example(1, 2, 3, 4, 5, kw_only='K', x=100, y=200)")
example(1, 2, 3, 4, 5, kw_only="K", x=100, y=200)

print("\nexample(1, pos_or_kw=2, kw_only='K')")
example(1, pos_or_kw=2, kw_only="K")

print("\nexample(1, 2, kw_only='K', pos_only=99)   # pos_only=99 — це НЕ параметр 1!")
example(1, 2, kw_only="K", pos_only=99)

print("\n--- Помилки ---")
calls = {
    "pos_only за іменем":   lambda: example(pos_only=1, pos_or_kw=2, kw_only="K"),
    "без kw_only":          lambda: example(1, 2),
}
for label, call in calls.items():
    try:
        call()
    except TypeError as e:
        print(f"  {label:<22} → TypeError: {e}")

💡 Зверніть увагу на третій виклик: `pos_only=99` **не** конфліктує з параметром `pos_only`. Той параметр — тільки позиційний, тому ім'я для іменованих аргументів вільне, і пара `pos_only=99` потрапляє в `kwargs`. Без `/` такий виклик дав би `TypeError: got multiple values`. Це ще одна причина, навіщо існує `/`: функція може приймати в `**kwargs` будь-які ключі, навіть ті, що збігаються з іменами її параметрів.

### 6.8 Основні проблеми та пастки обробки аргументів

#### Пастка 1: мутабельне значення за замовчуванням (Mutable Default Argument)

**Симптом:** функція «пам'ятає» попередні виклики.

**Чому:** `def` — це **виконувана інструкція**. Значення за замовчуванням обчислюється **ОДИН РАЗ** — коли виконується `def`, — і зберігається в атрибуті функції `__defaults__`. Усі виклики без цього аргументу отримують **той самий об'єкт**. Для `int`/`str` це непомітно (їх не можна змінити), а `list`/`dict`/`set` змінюються «на місці».

**Як правильно:** `None` як маркер «не передали» + новий об'єкт усередині тіла.

In [ ]:
# ❌ НЕПРАВИЛЬНО
def append_to(element, target=[]):
    print(f"    target id={id(target)}, до: {target}")
    target.append(element)
    return target


print(f"__defaults__ на старті: {append_to.__defaults__}")
print("append_to(1) →", append_to(1))
print("append_to(2) →", append_to(2))     # очікували [2]...
print(f"__defaults__ тепер:     {append_to.__defaults__}  ← список «наріс»")
print("append_to(3, []) →", append_to(3, []))   # свій список — інший id, пастки немає


# ✅ ПРАВИЛЬНО
def append_to_fixed(element, target=None):
    if target is None:
        target = []          # новий список при КОЖНОМУ виклику
    print(f"    target id={id(target)}, до: {target}")
    target.append(element)
    return target


print("\n--- з None ---")
print("append_to_fixed(1) →", append_to_fixed(1))
print("append_to_fixed(2) →", append_to_fixed(2))
print(f"__defaults__: {append_to_fixed.__defaults__}  ← не змінюється")

#### Пастка 2: передача «за посиланням на об'єкт» (Call-by-Sharing)

У Python аргументи не копіюються (не pass-by-value) і це не класичне C++-посилання (не pass-by-reference).
Параметр стає **ще одним ім'ям для того самого об'єкта**, на який вказує аргумент.

```
ззовні:   prices ──┐
                   ├──►  [100, 200]   ← ОДИН об'єкт у пам'яті
всередині:  lst  ──┘
```

| Дія всередині | Що відбувається | Видно ззовні? |
|---|---|---|
| `lst = [999]` | **перепризначення**: локальне ім'я тепер вказує на новий об'єкт | ❌ ні |
| `lst.append(99)`, `lst[0] = 1`, `lst += [5]` | **мутація** спільного об'єкта | ✅ так |

Слідкуйте за `id()` у виводі.

In [ ]:
def replace_val(lst):
    print(f"    [всередині до]    id={id(lst)}, lst={lst}")
    lst = [999]                      # нове ім'я → новий об'єкт
    print(f"    [всередині після] id={id(lst)}, lst={lst}  ← інший id")


def mutate_list(lst):
    print(f"    [всередині до]    id={id(lst)}, lst={lst}")
    lst.append(99)                   # змінюємо СПІЛЬНИЙ об'єкт
    print(f"    [всередині після] id={id(lst)}, lst={lst}  ← той самий id")


prices = [100, 200]
print(f"[ззовні] id={id(prices)}, prices={prices}")

print("\n--- replace_val(prices) ---")
replace_val(prices)
print(f"[ззовні] prices={prices}  ← не змінився")

print("\n--- mutate_list(prices) ---")
mutate_list(prices)
print(f"[ззовні] prices={prices}  ← ЗМІНИВСЯ!")

print("\n--- Захист: передати копію ---")
mutate_list(prices.copy())     # або list(prices), prices[:]
print(f"[ззовні] prices={prices}  ← оригінал цілий")

print("\n--- Незмінні типи (int, str, tuple) змінити «на місці» неможливо ---")
def add_one(n):
    n += 1          # для int це створює НОВИЙ об'єкт
    return n

x = 5
add_one(x)
print(f"x = {x}  ← int не змінюється; результат треба брати з return: {add_one(x)}")

#### Пастка 3: конфлікти аргументів під час виклику

1. **Одне значення двічі** — позиційно **і** за іменем → `TypeError` під час виконання.
2. **Позиційний після іменованого** → `SyntaxError` ще до запуску (тому демонструємо через `compile`).
3. **Невідоме ім'я** → `TypeError: ... unexpected keyword argument`.

In [ ]:
def foo(a, b):
    print(f"    a={a}, b={b}")


foo(1, b=2)          # ✅ спершу позиційні, потім іменовані
foo(b=2, a=1)        # ✅ іменовані — у будь-якому порядку

try:
    foo(1, a=2)
except TypeError as e:
    print(f"foo(1, a=2)   → TypeError: {e}")

try:
    foo(1, c=3)
except TypeError as e:
    print(f"foo(1, c=3)   → TypeError: {e}")

try:
    compile("foo(a=1, 2)", "<demo>", "exec")
except SyntaxError as e:
    print(f"foo(a=1, 2)   → SyntaxError: {e.msg}")

try:
    compile("foo(**{'a': 1}, *[2])", "<demo>", "exec")
except SyntaxError as e:
    print(f"foo(**d, *l)  → SyntaxError: {e.msg}")

#### Пастка 4: `**kwargs` ховає сигнатуру й ковтає помилки

`**kwargs` виглядає як універсальне рішення, але:
- IDE та перевірки типів (mypy) **не підкажуть**, які параметри існують;
- друкарська помилка (`debyg=True` замість `debug=True`) **мовчки** потрапляє в словник і ігнорується — або «вибухає» значно пізніше, глибоко в коді.

**Правило:** якщо ви знаєте імена параметрів — **напишіть їх явно** (keyword-only через `*`). `**kwargs` — для справжніх посередників (обгортки, декоратори, прокидання налаштувань далі).

In [ ]:
# ❌ Все через **kwargs
def run_report(data, **kwargs):
    debug = kwargs.get("debug", False)
    print(f"    kwargs={kwargs}, debug={debug}")
    if debug:
        print("    [DEBUG] детальний звіт")
    return sum(data)


print("run_report([1, 2, 3], debyg=True)  # описка!")
run_report([1, 2, 3], debyg=True)   # debug залишився False — і ЖОДНОЇ помилки


# ✅ Явний keyword-only параметр
def run_report_strict(data, *, debug=False):
    print(f"    debug={debug}")
    return sum(data)


print("\nrun_report_strict([1, 2, 3], debyg=True)")
try:
    run_report_strict([1, 2, 3], debyg=True)
except TypeError as e:
    print(f"    TypeError: {e}  ← описку видно одразу")

import inspect
print(f"\nСигнатури, як їх бачить IDE / help():")
print(f"  run_report{inspect.signature(run_report)}")
print(f"  run_report_strict{inspect.signature(run_report_strict)}")

### 6.9 Чому Python влаштований саме так?

1. **Усе є об'єктом (функції — об'єкти першого класу).**
   Функцію створює інструкція `def` — один раз, під час виконання. Значення за замовчуванням зберігаються в її атрибутах `__defaults__` (позиційні) і `__kwdefaults__` (keyword-only), код — у `__code__`. Обчислення один раз економить час на кожному виклику — ціна цього рішення і є Пастка 1.

2. **Продуктивність: дані не копіюються.**
   Модель Call-by-Sharing передає у функцію лише посилання. Список з мільйона елементів передається так само швидко, як число, — ціна цього рішення і є Пастка 2.

3. **Гнучкість: `*args` / `**kwargs` — фундамент метапрограмування.**
   Декоратори (урок 10), делегування `super().__init__(*args, **kwargs)` у класах, обгортки над чужими функціями — усе це можливо, бо функція може прийняти й передати далі аргументи, **не знаючи сигнатури наперед**.

4. **Еволюція API: `/` і `*` як контракт.**
   `/` (positional-only) дозволяє автору перейменовувати внутрішні параметри без злому чужого коду. `*` (keyword-only) змушує писати прапорці явно: `recv(1024, block=False)` читається набагато краще, ніж `recv(1024, False)`.

In [ ]:
def example(a, b=10, *args, flag=True, **kwargs):
    pass

print(f"__defaults__   = {example.__defaults__}")      # значення за замовчуванням позиційних
print(f"__kwdefaults__ = {example.__kwdefaults__}")    # значення за замовчуванням keyword-only
print(f"co_varnames    = {example.__code__.co_varnames}")
print(f"argcount       = {example.__code__.co_argcount}  (позиційні: a, b)")
print(f"kwonlyargcount = {example.__code__.co_kwonlyargcount}  (keyword-only: flag)")

---

## 🚪 Частина 7: Патерн — Предикат (Predicate / Filter)

**Ментальна модель: «Фейсконтроль»**

```
Охоронець (предикат) дивиться на кожен об'єкт і каже лише:
   ✓ TRUE  — пускає
   ✗ FALSE — не пускає

Предикат НІКОЛИ не змінює дані — тільки оцінює!
```

```
Вхід:  [1, 2, 3, 4, 5, 6, 7, 8]
Тест:  [✗, ✓, ✗, ✓, ✗, ✓, ✗, ✓]
Вихід: [2, 4, 6, 8]
```

### Три способи фільтрації в Python

```python
# 1. filter() — класичний
evens = list(filter(is_even, data))

# 2. list comprehension — пітонічний (найчастіше)
evens = [x for x in data if is_even(x)]

# 3. inline — без окремої функції
evens = [x for x in data if x % 2 == 0]
```

In [51]:
# Предикати — чисті функції що повертають True/False
def is_even(x):
    """Чи є число парним?"""
    return x % 2 == 0

def is_positive(x):
    """Чи є число додатним?"""
    return x > 0

def is_long_word(word):
    """Чи довше слово за 5 символів?"""
    return len(word) > 5


data = [1, -3, 2, -5, 4, 6, -1, 8]
words = ['cat', 'elephant', 'dog', 'python', 'ant', 'banana']

# Тест предикатів
print(f"is_even(4) = {is_even(4)},  is_even(3) = {is_even(3)}")
print(f"is_positive(-5) = {is_positive(-5)},  is_positive(3) = {is_positive(3)}")
print()

# Три способи фільтрації
evens_v1 = list(filter(is_even, data))               # filter()
evens_v2 = [x for x in data if is_even(x)]          # comprehension + предикат
evens_v3 = [x for x in data if x % 2 == 0]          # inline
print(f"data   = {data}")
print(f"filter()      → {evens_v1}")
print(f"comprehension → {evens_v2}")
print(f"inline        → {evens_v3}")
print()

long_words = [w for w in words if is_long_word(w)]
print(f"words      = {words}")
print(f"long words = {long_words}")

is_even(4) = True,  is_even(3) = False
is_positive(-5) = False,  is_positive(3) = True

data   = [1, -3, 2, -5, 4, 6, -1, 8]
filter()      → [2, 4, 6, 8]
comprehension → [2, 4, 6, 8]
inline        → [2, 4, 6, 8]

words      = ['cat', 'elephant', 'dog', 'python', 'ant', 'banana']
long words = ['elephant', 'python', 'banana']


In [52]:
# Комбінування предикатів — логічні оператори
data = [1, -3, 2, -5, 4, 6, -1, 8]

# AND: парні І додатні
positive_evens = [x for x in data if is_even(x) and is_positive(x)]
print(f"Парні і позитивні: {positive_evens}")

# OR: парні АБО додатні
even_or_positive = [x for x in data if is_even(x) or is_positive(x)]
print(f"Парні або позитивні: {even_or_positive}")

# NOT: непарні
odds = [x for x in data if not is_even(x)]
print(f"Непарні: {odds}")
print()

# all() та any() — редьюсери-предикати
scores = [85, 92, 78, 95, 88]
print(f"scores = {scores}")
print(f"all(s >= 70): {all(s >= 70 for s in scores)}  ← всі >= 70?")
print(f"any(s >= 90): {any(s >= 90 for s in scores)}  ← хоч один >= 90?")
print(f"all(s >= 90): {all(s >= 90 for s in scores)}  ← всі >= 90?")

Парні і позитивні: [2, 4, 6, 8]
Парні або позитивні: [1, 2, 4, 6, 8]
Непарні: [1, -3, -5, -1]

scores = [85, 92, 78, 95, 88]
all(s >= 70): True  ← всі >= 70?
any(s >= 90): True  ← хоч один >= 90?
all(s >= 90): False  ← всі >= 90?


---

## 🎨 Частина 8: Патерн — Трансформер (Transformer / Map)

**Ментальна модель: «Конвеєр фарбування»**

```
10 білих машин заїжджають → фарбувальна камера → 10 червоних виїжджають

Кількість НЕ змінюється: len(вхід) == len(вихід)
Змінюється тільки ФОРМА/СТАН кожного елемента
```

```
Вхід:  [1, 2, 3, 4, 5]           (5 елементів)
func:   x → x * x
Вихід: [1, 4, 9, 16, 25]         (5 елементів — завжди!)
```

### Generator Expression — лінива трансформація

```python
# List comprehension — всі дані одразу в пам'яті
squares = [x**2 for x in data]      # [1, 4, 9, 16, 25]

# Generator expression — обчислює по одному «на вимогу»
squares = (x**2 for x in data)      # <generator object>
# Ідеально для sum(), max(), min() — не витрачає пам'ять!
```

In [53]:
# Трансформери — функції що змінюють форму, не видаляють елементи
data = [1, 2, 3, 4, 5]

# 1. map() з lambda
squared_map = list(map(lambda x: x * x, data))

# 2. list comprehension (пітонічний стиль)
squared_comp = [x * x for x in data]

# 3. map() з іменованою функцією
def square(x): return x * x
squared_named = list(map(square, data))

print(f"data            = {data}")
print(f"map(lambda)     = {squared_map}")
print(f"comprehension   = {squared_comp}")
print(f"map(square)     = {squared_named}")
print(f"len(вхід) == len(вихід): {len(data) == len(squared_comp)}  ← завжди!")
print()

# Практичні приклади трансформацій
raw_names = ['  alice ', 'BOB', '  charlie  ']
clean_names = [name.strip().title() for name in raw_names]   # нормалізація
print(f"raw   = {raw_names}")
print(f"clean = {clean_names}")
print()

# Конвертація валюти
prices_usd = [10.5, 25.0, 3.99, 150.0]
rate       = 41.5
prices_uah = [round(p * rate, 2) for p in prices_usd]
print(f"USD: {prices_usd}")
print(f"UAH: {prices_uah}")

data            = [1, 2, 3, 4, 5]
map(lambda)     = [1, 4, 9, 16, 25]
comprehension   = [1, 4, 9, 16, 25]
map(square)     = [1, 4, 9, 16, 25]
len(вхід) == len(вихід): True  ← завжди!

raw   = ['  alice ', 'BOB', '  charlie  ']
clean = ['Alice', 'Bob', 'Charlie']

USD: [10.5, 25.0, 3.99, 150.0]
UAH: [435.75, 1037.5, 165.59, 6225.0]


In [8]:
import sys

# List comprehension vs Generator expression — пам'ять
lst_comp = [x**2 for x in range(10_000)]     # будує список в пам'яті
gen_expr = (x**2 for x in range(10_000))     # лінивий генератор

print(f"list comprehension: {sys.getsizeof(lst_comp):>8} байт")
print(f"generator expr    : {sys.getsizeof(gen_expr):>8} байт")
print(f"Різниця: x{sys.getsizeof(lst_comp)//sys.getsizeof(gen_expr):.0f} пам'яті")
print()

# Generator ідеальний для sum() — обчислює по одному
total_squared = sum(x**2 for x in range(10_000))   # gen всередині sum()
print(f"sum(x**2 for x in range(10000)) = {total_squared}")
print("  ↑ Generator не створює проміжний список — економить пам'ять!")

list comprehension:    85176 байт
generator expr    :      104 байт
Різниця: x819 пам'яті

sum(x**2 for x in range(10000)) = 333283335000
  ↑ Generator не створює проміжний список — економить пам'ять!


---

## ❄️ Частина 9: Патерн — Редьюсер (Reducer / Aggregation)

**Ментальна модель: «Снігова куля»**

```
Починається маленькою (початкове значення 0)
→ Котиться по масиві даних
→ Наліплює на себе кожен елемент
→ Перехід від БАГАТЬОХ до ОДНОГО

Вхід:  [10, 20, 30, 40]    → 4 елементи
       0+10=10
       10+20=30
       30+30=60
       60+40=100
Вихід: 100                  → одне значення
```

### Вбудовані редьюсери Python

| Функція | Операція | Опис |
|---------|----------|------|
| `sum()` | `0 + x1 + x2 + ...` | Сума |
| `max()` | найбільше | Максимум |
| `min()` | найменше | Мінімум |
| `all()` | AND по всіх | True якщо всі True |
| `any()` | OR по всіх | True якщо хоч один True |
| `functools.reduce()` | будь-яка | Для нестандартних операцій |

In [9]:
# Ручна реалізація — показуємо механіку
numbers = [10, 20, 30, 40, 50]

print("Крок | Накопичувач | Поточний елемент | Після додавання")
print("-" * 58)

accumulator = 0
for i, n in enumerate(numbers, 1):
    before = accumulator
    accumulator += n
    print(f"  {i}  | {before:11} | {n:16} | {accumulator}")

print(f"\nРезультат: {accumulator}")

Крок | Накопичувач | Поточний елемент | Після додавання
----------------------------------------------------------
  1  |           0 |               10 | 10
  2  |          10 |               20 | 30
  3  |          30 |               30 | 60
  4  |          60 |               40 | 100
  5  |         100 |               50 | 150

Результат: 150


In [10]:
import functools
import operator

data = [10, 20, 30, 40, 50]
print(f"data = {data}")
print()

# Вбудовані редьюсери — переважний стиль
print(f"sum(data)           = {sum(data)}")
print(f"max(data)           = {max(data)}")
print(f"min(data)           = {min(data)}")
print(f"len(data)           = {len(data)}")
print(f"sum/len (середнє)   = {sum(data)/len(data):.1f}")
print()

# Логічні редьюсери
print(f"all(x > 5 for x in data)  = {all(x > 5 for x in data)}   ← всі > 5?")
print(f"any(x > 40 for x in data) = {any(x > 40 for x in data)}  ← хоч один > 40?")
print()

# functools.reduce() — для нестандартних операцій
factorial = functools.reduce(operator.mul, range(1, 6))   # 1*2*3*4*5 = 120
print(f"reduce(mul, 1..5) = {factorial}  ← 5!")
print()

# max/min з ключем сортування
words = ['apple', 'banana', 'fig', 'cherry', 'kiwi']
longest  = max(words, key=len)
shortest = min(words, key=len)
print(f"words   = {words}")
print(f"найдовше: {longest!r}")
print(f"найкоротше: {shortest!r}")

data = [10, 20, 30, 40, 50]

sum(data)           = 150
max(data)           = 50
min(data)           = 10
len(data)           = 5
sum/len (середнє)   = 30.0

all(x > 5 for x in data)  = True   ← всі > 5?
any(x > 40 for x in data) = True  ← хоч один > 40?

reduce(mul, 1..5) = 120  ← 5!

words   = ['apple', 'banana', 'fig', 'cherry', 'kiwi']
найдовше: 'banana'
найкоротше: 'fig'


---

## 🔧 Частина 10: Функціональна Декомпозиція

**Стратегія «Розділяй і Володарюй» (Divide and Conquer)**

```
Велика задача
      ↓  декомпозиція
┌─────┴─────┐
│           │
Підзадача1  Підзадача2
│           │
функція1()  функція2()
```

**Принцип єдиної відповідальності:** одна функція = одна задача.
Якщо функцію важко назвати — вона, мабуть, робить забагато.

### Приклад: Знайти середню оцінку тільки тих, хто здав іспит

```
Задача: avg_passed_score(scores)
            ↓
   filter_passed(scores)     ← предикат: score >= 60
   calculate_average(passed) ← редьюсер: sum/len
```

---

## 🔄 Частина 11: Повний Pipeline — Filter → Map → Reduce

**Ідея пайплайну:** дані «протікають» через ланцюжок чистих функцій.

```
Сирі дані
    │
    ├─ Step 1: Predicate/Filter — відкинути зайве
    │
    ├─ Step 2: Transformer/Map  — змінити форму
    │
    └─ Step 3: Reducer/Reduce   — зібрати відповідь
```

**`*args` + generator = ефективний пайплайн:**
```python
def pipeline(*args):
    taxed = (sale * 1.10 for sale in args if sale > 0)  # filter + map у генераторі
    return sum(taxed)   # reduce — «тягне» дані з генератора по одному
```
- `sum()` — **консьюмер** (споживач), тягне дані
- Генератор — **продьюсер** (виробник), дає по одному

In [ ]:
# === Пайплайн з *args + generator ===

def process_sales_pipeline(*args):
    """
    Обчислює виручку з ПДВ тільки від позитивних продажів.

    *args = (150, -20, 300, 50, -10)
    Step 1 (Filter):  відкидаємо від'ємні
    Step 2 (Map):     застосовуємо ПДВ 10%
    Step 3 (Reduce):  сумуємо
    """
    taxed_gen = (sale * 1.10 for sale in args if sale > 0)
    #             ↑ MAP                ↑ FILTER
    return sum(taxed_gen)   # ← REDUCE


result = process_sales_pipeline(150, -20, 300, 50, -10)
print(f"Виручка з ПДВ (тільки позитивні): ${result:.2f}")
# Розрахунок: (150 + 300 + 50) * 1.10 = 550.0
print()

# Покроково — для розуміння
raw = (150, -20, 300, 50, -10)
step1 = [s for s in raw if s > 0]          # Filter
step2 = [s * 1.10 for s in step1]          # Map
step3 = sum(step2)                          # Reduce

print(f"Вхід:           {raw}")
print(f"Крок 1 (Filter): {step1}")
print(f"Крок 2 (Map):    {[round(x,2) for x in step2]}")
print(f"Крок 3 (Reduce): {step3:.2f}")

In [ ]:
# === Повний реальний пайплайн: аналіз оцінок класу ===

raw_data = [
    "Alice,92", "Bob,ERROR", "Charlie,85",
    "Diana,",   "Eve,78",    "Frank,65",
    "Grace,95", "Henry,71",  "Ivan,45",
]

print(f"Сирі дані: {len(raw_data)} рядків\n")

# Маленькі чисті функції
def is_valid_row(row):
    """Предикат: рядок має дві частини та числову оцінку."""
    parts = row.split(',')
    return len(parts) == 2 and parts[1].strip().isdigit()

def parse_row(row):
    """Трансформер: рядок → словник."""
    name, score = row.split(',')
    return {'name': name.strip(), 'score': int(score.strip())}

def add_grade(record):
    """Трансформер: додає літерну оцінку."""
    s = record['score']
    grade = 'A' if s>=90 else 'B' if s>=80 else 'C' if s>=70 else 'D' if s>=60 else 'F'
    return {**record, 'grade': grade}   # {**d, key: val} — копія з додатковим полем


# PIPELINE
# Step 1: Filter — тільки валідні рядки
valid_rows = [r for r in raw_data if is_valid_row(r)]

# Step 2: Map — парсимо у словники
records = [parse_row(r) for r in valid_rows]

# Step 3: Map — додаємо літерні оцінки
enriched = [add_grade(r) for r in records]

# Step 4: Filter — тільки ті хто здав (не F)
passed = [r for r in enriched if r['grade'] != 'F']

# Step 5: Reduce — агрегуємо
avg_all  = sum(r['score'] for r in enriched) / len(enriched)
avg_pass = sum(r['score'] for r in passed) / len(passed)

# Результат
print(f"Step 1 (Filter valid): {len(valid_rows)}/{len(raw_data)} рядків")
print(f"Step 5 (Passed):       {len(passed)}/{len(enriched)} студентів")
print(f"Середня по всіх:       {avg_all:.1f}")
print(f"Середня по здавших:    {avg_pass:.1f}")
print()
print("Таблиця студентів:")
print("-" * 32)
for r in enriched:
    marker = "✓" if r['grade'] != 'F' else "✗"
    print(f"  {marker} {r['name']:10}: {r['score']:3}  ({r['grade']})")

---

## ⚠️ Частина 12: Типові помилки — Застереження

### Застереження 1: `print` замість `return`
```python
# ❌ Функція нічого не повертає!
def add(a, b):
    print(a + b)      ← тільки показує на екрані

result = add(2, 3)   # result = None
result * 10          # TypeError!

# ✅ Повертаємо значення
def add(a, b):
    return a + b
```

### Застереження 2: Мутація вхідних даних (нечиста функція)
```python
# ❌ Знищує оригінал!
def add_item(lst, item):
    lst.append(item)   ← мутує оригінал
    return lst

# ✅ Чиста — зберігає оригінал
def add_item(lst, item):
    return lst + [item]  ← новий список
```

### Застереження 3: Глобальні змінні всередині функції
```python
# ❌ Прихована залежність — важко відлагодити
total = 0
def add_to_total(x):
    global total
    total += x   ← side effect!

# ✅ Параметри та return — явно
def add_to_total(current_total, x):
    return current_total + x
```

### Застереження 4: Надто велика функція (порушення декомпозиції)
```
❌ Якщо функція робить більше однієї речі — розбийте її!
   def process_and_validate_and_save():   ← три речі!

✅ Три окремі функції:
   def validate(data): ...
   def process(data): ...
   def save(data): ...
```

### Застереження 5: Мутабельне значення за замовчуванням
```python
# ❌ Список створюється ОДИН раз при def і накопичує стан
def add(item, items=[]): ...

# ✅ None як маркер + новий список усередині
def add(item, items=None):
    if items is None:
        items = []
```
Детально з логуванням `__defaults__` та `id()` — **Частина 6.8, Пастка 1**.

In [ ]:
# === Демонстрація застереження 2: мутація ===

# НЕЧИСТА — небезпечна!
def add_item_bad(lst, item):
    lst.append(item)   # мутує оригінал
    return lst

# ЧИСТА — безпечна
def add_item_good(lst, item):
    return lst + [item]   # новий список


original = [1, 2, 3]

result_bad = add_item_bad(original, 99)
print(f"нечиста: результат = {result_bad}")
print(f"нечиста: ОРИГІНАЛ ЗНИЩЕНИЙ = {original}")

print()

original2 = [1, 2, 3]
result_good = add_item_good(original2, 99)
print(f"чиста: результат = {result_good}")
print(f"чиста: оригінал збережено = {original2}")
print()

# === Застереження 3: global — антипатерн ===
print("=== Глобальні змінні — антипатерн ===")

total = 0   # глобальна — небезпечна

def bad_accumulate(x):
    global total       # прихована залежність!
    total += x

def good_accumulate(current, x):
    return current + x   # явно — без сюрпризів

bad_accumulate(10)
bad_accumulate(20)
print(f"bad_accumulate: total={total}  ← глобальна змінна змінена")

# Чистий варіант
running_total = 0
running_total = good_accumulate(running_total, 10)
running_total = good_accumulate(running_total, 20)
print(f"good_accumulate: total={running_total}  ← явний контроль")

---

## 📋 Частина 13: Шпаргалка (Quick Reference)

### Синтаксис
```python
def func_name(param1, param2=default, *args):   # визначення
    """Docstring — опис функції."""              # документація
    result = param1 + param2
    return result                                # передаємо в програму

answer = func_name(10, 20)                       # виклик
```

### Чотири патерни
```python
# PREDICATE (True/False)
def is_valid(x): return x > 0
valid = [x for x in data if is_valid(x)]   # filter
valid = list(filter(is_valid, data))        # альтернатива

# TRANSFORMER (один → інший)
def transform(x): return x * 1.10
mapped = [transform(x) for x in data]     # map
mapped = list(map(transform, data))        # альтернатива

# REDUCER (багато → одне)
total = sum(data)                          # сума
top   = max(data, key=func)               # максимум за ключем
check = all(x > 0 for x in data)          # всі?
found = any(x > 100 for x in data)        # хоч один?

# DECOMPOSE (велике → маленьке)
def step1_filter(data): ...
def step2_map(data): ...
def step3_reduce(data): ...
def full_pipeline(data):
    return step3_reduce(step2_map(step1_filter(data)))
```

### *args / **kwargs / порядок параметрів
```python
# ОГОЛОШЕННЯ — пакування
def func(*args, **kwargs):   # args — tuple, kwargs — dict
    ...
func(1, 2, x=3)              → args = (1, 2), kwargs = {'x': 3}

# ВИКЛИК — розпаковка
func(*[1, 2, 3])             → func(1, 2, 3)        # list / tuple / set / str / range
func(**{'x': 1, 'y': 2})     → func(x=1, y=2)       # dict → іменовані
func(*lst, **dct)            # спершу *, потім **

# ПОРЯДОК у def (порушення → SyntaxError)
def f(pos_only, /, pos_or_kw, default=10, *args, kw_only, kw_def=20, **kwargs): ...
#     └─ лише      └─ як               └─ зайві  └─ лише                └─ зайві
#        позиційно    завгодно            позиц.    key=value              іменовані

# ПОРЯДОК у виклику: спершу позиційні, потім іменовані; одне значення — один раз

# ПОЗА ФУНКЦІЯМИ
first, *rest = [1, 2, 3]     → first = 1, rest = [2, 3]
[*a, *b]  {*a, *b}  {**d1, **d2}    # склеїти / злити (правий ключ перемагає)

# ПАСТКА: items=[] за замовчуванням → items=None + if items is None: items = []
```

### Чисті функції
```python
# ✅ ЧИСТА: детерміновна + без side effects
def pure(lst): return sorted(lst)         # новий список

# ❌ НЕЧИСТА: мутує вхід або залежить від зовнішнього стану
def impure(lst): lst.sort(); return lst   # in-place!
```

---

# 📝 Задачі — Самостійна робота

### TODO 1: Чиста функція vs Нечиста

Дано список цін. Напишіть **чисту функцію** `apply_vat(prices, rate=0.20)`,
яка повертає **новий** список цін з ПДВ.
Перевірте що оригінал залишається незмінним.

In [ ]:
prices = [100.0, 250.0, 75.0, 400.0]

# YOUR CODE HERE
def apply_vat(prices, rate=0.20):
    pass   # замініть на реалізацію

result = apply_vat(prices)
print(f"Результат: {result}")
print(f"Оригінал:  {prices}")   # має бути незмінним!

# Очікуваний вивід:
# Результат: [120.0, 300.0, 90.0, 480.0]
# Оригінал:  [100.0, 250.0, 75.0, 400.0]

### TODO 2: Предикат + Фільтрація

Дано список слів. Напишіть предикат `is_palindrome(word)` (True якщо слово-паліндром).
Знайдіть усі палідроми у списку.

In [ ]:
words = ['level', 'python', 'radar', 'hello', 'civic', 'world', 'madam', 'noon']

# YOUR CODE HERE
def is_palindrome(word):
    pass   # Підказка: word == word[::-1]

# Знайдіть усі палідроми
# palindromes = ...

# Очікуваний вивід:
# ['level', 'radar', 'civic', 'madam', 'noon']

### TODO 3: *args Редьюсер

Напишіть функцію `sales_stats(*amounts)`, яка приймає
довільну кількість сум і повертає словник:
`{'count': ..., 'total': ..., 'average': ..., 'max': ...}`

In [ ]:
# YOUR CODE HERE
def sales_stats(*amounts):
    pass   # amounts — це tuple всередині функції

stats = sales_stats(150, 300, 75, 200, 450)
print(stats)

# Очікуваний вивід:
# {'count': 5, 'total': 1175, 'average': 235.0, 'max': 450}

### TODO 3b: `*args` + `**kwargs` — замовлення в кафе

Напишіть функцію `make_order(table, *dishes, **options)`, яка повертає словник замовлення.
- `dishes` — довільна кількість страв (tuple → перетворіть у list);
- `options` — довільні налаштування (`takeaway=True`, `comment="без цукру"` …);
- додайте `print`-логування: що потрапило в `table`, `dishes`, `options`.

Потім викличте її **з розпаковкою** готового списку страв і словника опцій.

In [ ]:
# YOUR CODE HERE
def make_order(table, *dishes, **options):
    pass

menu_choice = ["кава", "круасан"]
extras = {"takeaway": False, "comment": "без цукру"}

order = make_order(5, *menu_choice, **extras)
print(order)

# Очікуваний вивід (після логування):
# {'table': 5, 'dishes': ['кава', 'круасан'], 'options': {'takeaway': False, 'comment': 'без цукру'}}

### TODO 4: Трансформер Pipeline

Дано список рядків з числами, деякі — з помилками.
Побудуйте пайплайн:
1. Відфільтруйте валідні (`.isdigit()`)
2. Перетворіть у числа (`int`)
3. Залишіть тільки чотиризначні (1000–9999)
4. Відсортуйте за спаданням
5. Виведіть середнє

In [ ]:
raw = ['1234', 'abc', '5678', '99', 'ERROR', '4321', '10000', '8888', '', '7654']

# YOUR CODE HERE
# Step 1: filter valid strings
# Step 2: map to int
# Step 3: filter 4-digit
# Step 4: sort descending
# Step 5: calculate average

# Очікуваний вивід:
# Відфільтровано: [8888, 7654, 5678, 4321, 1234]
# Середнє: 5515.0

### TODO 5 ⭐: Повна декомпозиція

Дано список транзакцій: `(категорія, сума)`.
Використовуючи декомпозицію (маленькі чисті функції), напишіть:
1. `filter_positive(txs)` — тільки позитивні суми
2. `group_by_category(txs)` — `{категорія: [суми]}`
3. `category_totals(groups)` — `{категорія: сума}`
4. `top_category(totals)` — категорія з найбільшою сумою
5. Зберіть у `full_report(txs)` що викликає всі чотири

In [ ]:
from collections import defaultdict

transactions = [
    ('Food', 150), ('Transport', 30), ('Food', -20),
    ('Books', 80), ('Food', 200),     ('Transport', -5),
    ('Books', 120), ('Food', 90),     ('Transport', 45),
]

# YOUR CODE HERE
def filter_positive(txs): pass
def group_by_category(txs): pass
def category_totals(groups): pass
def top_category(totals): pass
def full_report(txs): pass

# Очікуваний вивід:
# Категорії: {'Food': 440, 'Transport': 75, 'Books': 200}
# Топ-категорія: Food